In [2]:
import torch
from dataExtraction import get_proteomics, get_genes
from utils import run_experiment
from models import LinearModel, OneHiddenLayer
from losses import VariancePenaltyLoss
import torch.nn as nn
import pandas as pd
import numpy as np

# Checking for Metal availabilty
if torch.mps.is_available():
    device = torch.device("mps")
    print("M4 GPU acceleration with Metal is ready")
else:
    device = torch.device("cpu")
    print("Metal not found, back on cpu")

M4 GPU acceleration with Metal is ready


In [ ]:
CONFIG = {
    "n_top_genes"  : 300,
    "n_splits"     : 5,
    "batch_size"   : 128,
    "lr"           : 1e-3,
    "weight_decay" : 0.01,
    "n_neighbors"  : 5,
    "max_epochs"   : 500,
    "patience"     : 20,
    "min_delta"    : 1e-4,
    "device"       : device,
}


In [4]:
# Data Loading
X_prot, X_meta       = get_proteomics("data/sound_life_all_olink.csv", only_day_0=False)
Y, gene_names, ids   = get_genes("data/sound-life_whole-blood_no-stim.h5ad", X_prot, n_top_genes=CONFIG["n_top_genes"])

proteomics_raw    = X_prot.loc[ids].values
metadata          = X_meta.loc[ids]
sex_encoded       = pd.get_dummies(metadata['subject.biologicalSex'], drop_first=True)
metadata_numeric  = pd.concat([sex_encoded, metadata['sample.subjectAgeAtDraw']], axis=1).astype(float).values
groups            = pd.factorize(metadata['subject.subjectGuid'])[0]

/Users/paulmaricelle/Stanford/mini-projet/dataExtraction.py:7: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


There are : 32 samples in X without corresponding value in Y


In [17]:
# Experiments : They require one model and one loss
from functools import partial

experiments = {
    "Linear + MSE"          : (LinearModel,    nn.MSELoss()),
    "Linear + VarPenalty"   : (LinearModel,    VariancePenaltyLoss(alpha=0.5)),
    "OneHidden + MSE"       : (partial(OneHiddenLayer, hidden=4096), nn.MSELoss()),
    "OneHidden + VarPenalty": (partial(OneHiddenLayer, hidden=4096), VariancePenaltyLoss(alpha=0.5)),
}

In [18]:
# Computing results and displaying them as a spreadsheet
results = {}
for name, (model_class, loss_fn) in experiments.items():
    print(f"\n{'='*50}\n{name}\n{'='*50}")
    results[name] = run_experiment(model_class, loss_fn, proteomics_raw, Y, metadata_numeric, groups, CONFIG)

print(f"\n{'Experiment':<30} | {'Median r':>10} | {'Ratio std':>10}")
print("-" * 55)
for name, scores in results.items():
    median_r  = np.mean(scores["median_r"])
    std_r     = np.std(scores["median_r"])
    ratio_std = np.mean(scores["ratio_std"])
    print(f"{name:<30} | {median_r:.3f} ± {std_r:.3f} | {ratio_std:.3f}")


Linear + MSE

--- Fold 1/5 ---
Epoch    1 | Train loss : 0.8038 | Val loss : 0.7095 | No improve : 0/20
Epoch   10 | Train loss : 0.2214 | Val loss : 0.5509 | No improve : 0/20
Epoch   20 | Train loss : 0.2184 | Val loss : 0.5442 | No improve : 2/20
Epoch   30 | Train loss : 0.2182 | Val loss : 0.5404 | No improve : 6/20
Epoch   40 | Train loss : 0.2175 | Val loss : 0.5417 | No improve : 16/20
Early stopping at epoch 43 — best val loss : 0.5252

--- Fold 2/5 ---
Epoch    1 | Train loss : 0.8235 | Val loss : 0.7060 | No improve : 0/20
Epoch   10 | Train loss : 0.2239 | Val loss : 0.5562 | No improve : 1/20
Epoch   20 | Train loss : 0.2172 | Val loss : 0.5558 | No improve : 6/20
Epoch   30 | Train loss : 0.2181 | Val loss : 0.5580 | No improve : 16/20
Early stopping at epoch 33 — best val loss : 0.5505

--- Fold 3/5 ---
Epoch    1 | Train loss : 0.8118 | Val loss : 0.8296 | No improve : 0/20
Epoch   10 | Train loss : 0.2177 | Val loss : 0.6429 | No improve : 0/20
Epoch   20 | Train loss